# Qwen2-7B MAP Training Notebook

This notebook is structured for Kaggle + H100.

- Start with `QUICK_RUN = True` to sanity-check the pipeline.
- Switch to `QUICK_RUN = False` for full training.
- The trained adapter is saved under the model's own name.
- Quick run prints validation `MAP@3`.


In [ ]:
!pip install -q -U "transformers>=4.55.0" peft accelerate bitsandbytes datasets scikit-learn joblib


In [ ]:
import gc
import inspect
import json
import os
import random
import re
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
import joblib
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

TEMP_DIR = Path("/tmp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

TRAIN_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"
OUTPUT_ROOT = Path("/kaggle/working/map_multi_llm_runs")

# If you have a Kaggle-mounted local model folder, it will be used.
# Otherwise the notebook falls back to the public Hugging Face base model.
LOCAL_MODEL_PATH = "/kaggle/input/qwen-2-5-map/transformers/default/1"
HF_FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MODEL_NAME = LOCAL_MODEL_PATH if Path(LOCAL_MODEL_PATH).exists() else HF_FALLBACK_MODEL_NAME

QUICK_RUN = True
QUICK_RUN_ROWS = 2048
QUICK_RUN_TEST_SIZE = 0.10
QUICK_RUN_EPOCHS = 1
QUICK_RUN_MAX_STEPS = 40
FULL_TRAIN_EPOCHS = 2

USE_4BIT = True
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

MAX_LENGTH = 256
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.1
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4

LORA_R = 64
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MODULES_TO_SAVE = ["score"]

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_hf_token():
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def resolve_model_source(model_name: str, hf_fallback_model_name: str | None = None):
    candidate = Path(model_name)
    if candidate.exists() and candidate.is_dir() and (candidate / "config.json").exists():
        return str(candidate.resolve()), {"local_files_only": True}, "local_path"

    if candidate.exists() and candidate.is_dir():
        nested = sorted(
            path for path in candidate.rglob("*")
            if path.is_dir() and (path / "config.json").exists()
        )
        if nested:
            chosen = nested[0]
            print(f"Resolved nested model folder: {chosen}")
            return str(chosen.resolve()), {"local_files_only": True}, "local_path_discovered"

    looks_like_path = model_name.startswith("/") or "\\" in model_name or ":" in model_name
    if looks_like_path:
        parent = candidate.parent
        if parent.exists():
            version_dirs = [
                path for path in parent.iterdir()
                if path.is_dir() and (path / "config.json").exists()
            ]
            if version_dirs:
                version_dirs = sorted(
                    version_dirs,
                    key=lambda path: (not path.name.isdigit(), int(path.name) if path.name.isdigit() else path.name),
                )
                chosen = version_dirs[-1]
                print(f"Requested model path not found, using available version: {chosen}")
                return str(chosen.resolve()), {"local_files_only": True}, "local_path_auto_discovered"

        search_root = Path("/kaggle/input")
        discovered = []
        if search_root.exists():
            for path in sorted(search_root.glob("*/transformers/default/*")):
                if path.is_dir() and (path / "config.json").exists():
                    discovered.append(str(path))

        details = ""
        if discovered:
            preview = "\n- " + "\n- ".join(discovered[:10])
            details = f"\nAvailable Kaggle model folders:{preview}"

        if hf_fallback_model_name:
            print(
                "Local model path was not found. "
                f"Falling back to Hugging Face model: {hf_fallback_model_name}"
            )
            if details:
                print(details.lstrip("\n"))
            return hf_fallback_model_name, {}, "hf_hub_fallback"

        raise FileNotFoundError(
            f"Local model path does not exist: {model_name}{details}\n"
            "Attach the correct Kaggle model folder and update MODEL_NAME to the directory that contains config.json."
        )

    return model_name, {}, "hf_hub"


def make_model_slug(name_or_path: str) -> str:
    raw = name_or_path.rstrip("/")
    parts = Path(raw).parts
    if "transformers" in parts:
        idx = parts.index("transformers")
        base = parts[idx - 1] if idx > 0 else Path(raw).name
    elif "/" in raw:
        base = raw.split("/")[-1]
    else:
        base = Path(raw).name or "model"
    return re.sub(r"[^A-Za-z0-9._-]+", "_", base)


def format_input_v2(row):
    x = "Yes"
    if not row["is_correct"]:
        x = "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Is Correct Answer: {x}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )


def compute_map3(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    top3 = np.argsort(-probs, axis=1)[:, :3]
    match = top3 == labels[:, None]

    map3 = 0.0
    for i in range(len(labels)):
        if match[i, 0]:
            map3 += 1.0
        elif match[i, 1]:
            map3 += 1.0 / 2.0
        elif match[i, 2]:
            map3 += 1.0 / 3.0

    return {"map@3": map3 / len(labels)}


def build_training_frame(df: pd.DataFrame):
    df = df.copy()
    df["Misconception"] = df["Misconception"].fillna("NA")
    df["target"] = df["Category"] + ":" + df["Misconception"]

    le = LabelEncoder()
    df["label"] = le.fit_transform(df["target"])
    n_classes = len(le.classes_)

    idx = df.apply(lambda row: row.Category.split("_")[0], axis=1) == "True"
    correct = df.loc[idx].copy()
    correct["c"] = correct.groupby(["QuestionId", "MC_Answer"]).MC_Answer.transform("count")
    correct = correct.sort_values("c", ascending=False)
    correct = correct.drop_duplicates(["QuestionId"])
    correct = correct[["QuestionId", "MC_Answer"]]
    correct["is_correct"] = 1

    df = df.merge(correct, on=["QuestionId", "MC_Answer"], how="left")
    df.is_correct = df.is_correct.fillna(0).astype(np.int64)
    df["text"] = df.apply(format_input_v2, axis=1)
    return df, le, n_classes


def dataframe_to_dataset(frame: pd.DataFrame, tokenizer, max_length: int) -> Dataset:
    clean = frame[["text", "label"]].copy()
    clean["label"] = clean["label"].astype(np.int64)
    ds = Dataset.from_pandas(clean, preserve_index=False)
    ds = ds.rename_column("label", "labels")

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    return ds.map(tokenize_fn, batched=True, remove_columns=["text"])


seed_everything(SEED)
HF_TOKEN = get_hf_token()
RESOLVED_MODEL_NAME, MODEL_LOAD_EXTRA_KWARGS, MODEL_SOURCE_TYPE = resolve_model_source(
    MODEL_NAME,
    hf_fallback_model_name=HF_FALLBACK_MODEL_NAME,
)
MODEL_SLUG = make_model_slug(RESOLVED_MODEL_NAME)
RUN_NAME = "quick_run" if QUICK_RUN else "full_train"
RUN_DIR = OUTPUT_ROOT / MODEL_SLUG / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
FINAL_MODEL_DIR = RUN_DIR / MODEL_SLUG

for path in [OUTPUT_ROOT, RUN_DIR, CHECKPOINT_DIR, LOG_DIR, FINAL_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print({
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "model_name": MODEL_NAME,
    "resolved_model_name": RESOLVED_MODEL_NAME,
    "model_source_type": MODEL_SOURCE_TYPE,
    "quick_run": QUICK_RUN,
    "use_4bit": USE_4BIT,
    "model_dtype": str(MODEL_DTYPE).replace("torch.", ""),
    "run_dir": str(RUN_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
})


In [ ]:
train_full = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_full, le, n_classes = build_training_frame(train_full)
id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in id2label.items()}

print(f"Train shape: {train_full.shape} with {n_classes} target classes")
print("Train head:")
print(train_full.head())
print(f"test_df shape: {test_df.shape}")
print("\nExample prompt for our LLM (after refinement):")
print(train_full.text.values[0])

work_df = train_full.copy()
if QUICK_RUN:
    sample_size = min(QUICK_RUN_ROWS, len(work_df))
    work_df = work_df.sample(n=sample_size, random_state=SEED).reset_index(drop=True)
    stratify_labels = work_df["label"] if work_df["label"].value_counts().min() >= 2 else None
    train_df, eval_df = train_test_split(
        work_df,
        test_size=QUICK_RUN_TEST_SIZE,
        random_state=SEED,
        stratify=stratify_labels,
    )
else:
    train_df, eval_df = work_df, None

train_df = train_df.reset_index(drop=True)
if eval_df is not None:
    eval_df = eval_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Epochs:", QUICK_RUN_EPOCHS if QUICK_RUN else FULL_TRAIN_EPOCHS)
print("Batch size:", PER_DEVICE_TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Validation rows:", 0 if eval_df is None else len(eval_df))


In [ ]:
model_load_kwargs = {
    "trust_remote_code": True,
    "cache_dir": str(TEMP_DIR),
    **MODEL_LOAD_EXTRA_KWARGS,
}
if HF_TOKEN:
    model_load_kwargs["token"] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(
    RESOLVED_MODEL_NAME,
    **model_load_kwargs,
)

added_pad_token = False
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    added_pad_token = True
tokenizer.padding_side = "right"

train_ds = dataframe_to_dataset(train_df, tokenizer, MAX_LENGTH)
eval_ds = dataframe_to_dataset(eval_df, tokenizer, MAX_LENGTH) if eval_df is not None else None
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(train_ds)
print(eval_ds)


In [ ]:
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

base_model_kwargs = {
    "pretrained_model_name_or_path": RESOLVED_MODEL_NAME,
    "num_labels": n_classes,
    "quantization_config": bnb_config,
    "device_map": "auto",
    "cache_dir": str(TEMP_DIR),
    "torch_dtype": MODEL_DTYPE,
    "trust_remote_code": True,
    **MODEL_LOAD_EXTRA_KWARGS,
}
if HF_TOKEN:
    base_model_kwargs["token"] = HF_TOKEN

model = AutoModelForSequenceClassification.from_pretrained(**base_model_kwargs)

if added_pad_token:
    model.resize_token_embeddings(len(tokenizer))

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.id2label = id2label
model.config.label2id = label2id
model.config.problem_type = "single_label_classification"

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=MODULES_TO_SAVE,
)

model = prepare_model_for_kbit_training(model) if USE_4BIT else model
model = get_peft_model(model, lora_config)
model = model.to(dtype=torch.bfloat16 if USE_BF16 else MODEL_DTYPE)

print(next(model.parameters()).dtype)
model.print_trainable_parameters()


In [ ]:
epochs = QUICK_RUN_EPOCHS if QUICK_RUN else FULL_TRAIN_EPOCHS
max_steps = QUICK_RUN_MAX_STEPS if QUICK_RUN else -1

training_arg_values = {
    "output_dir": str(CHECKPOINT_DIR),
    "do_train": True,
    "do_eval": eval_ds is not None,
    "evaluation_strategy": "steps" if eval_ds is not None else "no",
    "eval_steps": 10 if eval_ds is not None else None,
    "save_strategy": "no",
    "num_train_epochs": epochs,
    "max_steps": max_steps,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "logging_dir": str(LOG_DIR),
    "logging_steps": 10 if QUICK_RUN else 100,
    "remove_unused_columns": False,
    "bf16": USE_BF16,
    "fp16": torch.cuda.is_available() and not USE_BF16,
    "report_to": "none",
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": "cosine",
    "dataloader_drop_last": True,
    "dataloader_pin_memory": False,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    "seed": SEED,
    "disable_tqdm": True,
}

training_arg_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_arg_signature and "evaluation_strategy" not in training_arg_signature:
    training_arg_values["eval_strategy"] = training_arg_values.pop("evaluation_strategy")

supported_training_args = {
    key: value
    for key, value in training_arg_values.items()
    if key in training_arg_signature and value is not None
}

training_args = TrainingArguments(**supported_training_args)
trainer_signature = inspect.signature(Trainer.__init__).parameters

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "data_collator": data_collator,
}

if eval_ds is not None:
    trainer_kwargs["eval_dataset"] = eval_ds
    trainer_kwargs["compute_metrics"] = compute_map3

if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

supported_trainer_kwargs = {
    key: value
    for key, value in trainer_kwargs.items()
    if key in trainer_signature and value is not None
}

trainer = Trainer(**supported_trainer_kwargs)

for callback in list(trainer.callback_handler.callbacks):
    if callback.__class__.__name__ == "NotebookProgressCallback":
        trainer.remove_callback(callback.__class__)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()
metrics = dict(train_result.metrics)

if eval_ds is not None:
    metrics.update(trainer.evaluate())

print("Saving model and label encoder...")
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
joblib.dump(le, RUN_DIR / "label_encoder.joblib")

summary = {
    "model_name": MODEL_NAME,
    "resolved_model_name": RESOLVED_MODEL_NAME,
    "model_source_type": MODEL_SOURCE_TYPE,
    "quick_run": QUICK_RUN,
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "run_dir": str(RUN_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
    "n_classes": int(n_classes),
    "train_rows": int(len(train_df)),
    "eval_rows": int(0 if eval_df is None else len(eval_df)),
    "metrics": metrics,
}

with open(RUN_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved adapter to: {FINAL_MODEL_DIR}")
print(f"Saved label encoder to: {RUN_DIR / 'label_encoder.joblib'}")
if eval_ds is not None:
    print(f"Validation MAP@3: {metrics.get('eval_map@3')}")
print(json.dumps(metrics, indent=2))


In [ ]:
reload_bnb_config = None
if USE_4BIT:
    reload_bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

reload_model_kwargs = dict(base_model_kwargs)
reload_model_kwargs["quantization_config"] = reload_bnb_config

infer_base_model = AutoModelForSequenceClassification.from_pretrained(**reload_model_kwargs)
infer_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR), trust_remote_code=True)
infer_model = PeftModel.from_pretrained(infer_base_model, str(FINAL_MODEL_DIR))
infer_model.eval()

sample_text = train_df["text"].iloc[0]
inputs = infer_tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
model_device = next(infer_model.parameters()).device
inputs = {k: v.to(model_device) for k, v in inputs.items()}

with torch.no_grad():
    logits = infer_model(**inputs).logits
    top3 = torch.topk(logits, k=min(3, logits.shape[-1]), dim=-1).indices[0].detach().cpu().numpy()

print("Sample top-3 predicted labels:")
print(le.inverse_transform(top3))
